# 09 · `gl_engine/interp/tree.py`

## What this file is for

ISO's rules don't take arguments; they read and write a **shared data tree**. A submission goes in as a tree, every rule reads from it by path, and the premium is a value written back onto it.

This file is that tree and the little path language that addresses it. If you understand this, the interpreter stops looking mysterious: it is a program whose entire memory is one document.

**Depends on:** [`08-interp-values`](08-interp-values.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.interp import tree

for name, obj in vars(tree).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != tree.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Build a small tree and read something out of it.

In [ ]:
from gl_engine.interp import tree as T

node = T.Node.from_dict("GeneralLiability", {
    "StateCode": "GA",
    "EffDate": "20260811",
    "GeneralLiabilityLocation": [
        {"LocationNumber": "1", "PremisesOperationsTerr": "001"},
        {"LocationNumber": "2", "PremisesOperationsTerr": "002"},
    ],
})

print(T.dump(node))

## The interesting case

### Reading by path

In [ ]:
print("StateCode          :", T.read("StateCode", node))
print("first location terr:", T.read("GeneralLiabilityLocation/PremisesOperationsTerr", node))
print()
hits = T.select("GeneralLiabilityLocation", node)
print(f"select returns every match: {len(hits)} locations")
for h in hits:
    print("   location", T.read("LocationNumber", h), "->", T.read("PremisesOperationsTerr", h))

`select` returning a **list** rather than one node is the whole reason multi-location risks work. ISO's `ForEach` walks exactly this, and a reader that quietly took the first match would price a five-location risk as a one-location risk — with no error anywhere.

### Writing, and `ensure`

Rules write their results back. `ensure` creates the path if it isn't there yet, which is how a rule can populate somewhere nothing has written before.

In [ ]:
T.write("TotalPremium", node, "7366")
print("after write:", T.read("TotalPremium", node))

made = T.ensure("Coverages/Products/BasicLimitPremium", node)
print("ensure built:", made.tag)
T.write("Coverages/Products/BasicLimitPremium", node, "6845")
print("read back   :", T.read("Coverages/Products/BasicLimitPremium", node))

### Context is relative

Every path is evaluated against a context node, not against the root. That is what lets one rule run once per location without knowing which location it is on.

In [ ]:
for loc in T.select("GeneralLiabilityLocation", node):
    terr = T.read("PremisesOperationsTerr", loc)      # relative to THIS location
    print(f"location {T.read('LocationNumber', loc)} sees territory {terr}")

print()
print("root from a leaf:", T.select("GeneralLiabilityLocation", node)[0].root.tag)

## What it refuses

A path that matches nothing reads as `None` — it does not invent a node.

In [ ]:
print("missing path       :", repr(T.read("NoSuchField", node)))
print("select on nothing  :", T.select("NoSuchField", node))
print("select_one         :", repr(T.select_one("NoSuchField", node)))
print()
print("Reading absent data is None, not 0 and not an error --")
print("which is exactly the null from notebook 08 travelling up.")

## Try it yourself

1. Build a tree with three locations and two classifications under each. What does `select` return for the classifications from the root?
2. What happens if you `write` to a path that partly exists? Where does it stop and start creating?
3. Print the tree of a real rated submission — [`16-rating-kernel`](16-rating-kernel.ipynb) shows how to get one.

In [ ]:
# your turn